# Data Analysis & Modeling: Predict Whether a Player Joins an Experience

**Original interview mock, not a reported Roblox question or an official CodeSignal assessment.**
Interview format: **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. All data are synthetic.

## The interviewer prompt

At the time an experience card is shown, predict whether the player joins that experience within the next ten minutes. Each row is one displayed card (an impression), not one player.

**Problem 1 (14 minutes):** inspect and clean impression logs, attach experience metadata, and report join rate by genre.
**Problem 2 (21 minutes):** build a probability model, use a user-disjoint holdout, evaluate imbalanced classification, and score new cards.

This is a small pointwise recommendation baseline, not a full recommendation system. Ranking cards is an optional follow-up after the timed exercise.

| Coding time | Task |
| --- | --- |
| 0–4 min | Load and inspect |
| 4–9 min | Clean the data |
| 9–14 min | Analyze and prepare features |
| 14–18 min | Select features and split |
| 18–27 min | Preprocess and train |
| 27–32 min | Evaluate |
| 32–35 min | Predict and summarize |

These are suggested checkpoints, not official sub-round timings. Ask during the introduction: What is one example? When is a prediction made? What exactly is the target?

**Reference version.** Run cells top to bottom. Code is complete; each step includes checks and expandable follow-up answers. Try the matching practice notebook first.

## Data contract — supplied by the interviewer

The interviewer supplies **impressions.csv** and **experiences.csv** in **data/codesignal_roblox_recommendation/**. Load these into DataFrames named **impressions** and **experiences** in Step 1. Each user has one request with four shown candidates. All ten-minute outcomes are observed. Historical affinity was computed before the request; experience attributes are snapshots available at request time.

| Column | Meaning / policy |
| --- | --- |
| impression_id | Unique displayed-card ID, except exact repeated exports |
| user_id, request_id, experience_id | Group/join identifiers, not numeric features |
| device | Device at request time, may be missing |
| prior_genre_minutes | User's minutes in that genre during the previous seven days; negatives mean unknown |
| friends_playing | Number of friends playing at request time |
| position | Display position 1–4; affects observed behavior |
| joined | 0/1 label: join this experience within ten minutes |
| minutes_after_impression | Subsequent play time; unavailable for the current prediction |
| experiences.genre, experience_age_days | Static/as-of-request experience metadata |

The evaluation question is explicitly **generalization to unseen users**. Users must not cross the train/test boundary. Multiple joins per request are possible over ten minutes; this is not a mutually exclusive softmax task. Logs contain only displayed candidates, not all possible experiences.


## Provided setup

Open this notebook from the workspace root in VS Code or Jupyter and select **.venv-codesignal/bin/python**. No network, GPU or additional package installation is required.

The two input files are **data/codesignal_roblox_recommendation/impressions.csv** and **data/codesignal_roblox_recommendation/experiences.csv**. Load them into impressions and experiences in Step 1. The source files are already supplied; data generation is not part of the exercise.

The next cell supplies imports and DATA_DIR. Start the 35-minute coding timer after setup.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score,
    accuracy_score, f1_score, log_loss, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, precision_score, recall_score)
SEED = 42

DATA_DIR = Path("data/codesignal_roblox_recommendation")
assert (DATA_DIR / "impressions.csv").exists(), "Run from the workspace root."
assert (DATA_DIR / "experiences.csv").exists()


## Problem 1 · Step 1 — Load and inspect (0–4 min)

Load impressions.csv and experiences.csv into impressions and experiences with pd.read_csv. Inspect both tables. Count users, requests, duplicate impressions, missingness and joined class balance.

**Say before coding:** "One example is a displayed user-experience pair. Repeated users make a naive random row split risky for our unseen-user evaluation."

**Possible interviewer questions**

1. What does one training example represent?
2. Are unshown items negative examples?
3. What should you inspect before dropping missing rows?


In [2]:
impressions = pd.read_csv(DATA_DIR / "impressions.csv")
experiences = pd.read_csv(DATA_DIR / "experiences.csv")
print("impressions:", impressions.shape, "experiences:", experiences.shape)
impressions.info()
display(impressions.head(), experiences.head(), impressions.isna().sum())
print("users:", impressions["user_id"].nunique())
print("requests:", impressions["request_id"].nunique())
print("exact duplicate rows:", impressions.duplicated().sum())
display(impressions["joined"].value_counts(normalize=True))


impressions: (2420, 10) experiences: (60, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2420 entries, 0 to 2419
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   impression_id             2420 non-null   int64  
 1   user_id                   2420 non-null   int64  
 2   request_id                2420 non-null   int64  
 3   experience_id             2420 non-null   int64  
 4   device                    2370 non-null   object 
 5   position                  2420 non-null   int64  
 6   prior_genre_minutes       2420 non-null   float64
 7   friends_playing           2420 non-null   int64  
 8   joined                    2420 non-null   int64  
 9   minutes_after_impression  2420 non-null   float64
dtypes: float64(2), int64(7), object(1)
memory usage: 189.2+ KB


,impression_id,user_id,request_id,experience_id,device,position,prior_genre_minutes,friends_playing,joined,minutes_after_impression
0,0,0,0,51,desktop,1,27.028748,1,1,20.215565
1,1,0,0,37,desktop,2,47.208345,1,1,13.380891
2,2,0,0,38,desktop,3,26.547296,1,0,0.000000
3,3,0,0,27,desktop,4,17.268351,0,0,0.000000
4,4,1,1,41,mobile,1,23.604279,1,0,0.000000


,experience_id,genre,experience_age_days
0,0,adventure,1383
1,1,puzzle,1117
2,2,simulation,550
3,3,simulation,1451
4,4,simulation,616


impression_id                0
user_id                      0
request_id                   0
experience_id                0
device                      50
position                     0
prior_genre_minutes          0
friends_playing              0
joined                       0
minutes_after_impression     0
dtype: int64

users: 600
requests: 600
exact duplicate rows: 20


joined
0    0.919008
1    0.080992
Name: proportion, dtype: float64

In [3]:
assert experiences["experience_id"].is_unique
assert impressions["experience_id"].isin(experiences["experience_id"]).all()
assert set(impressions["joined"]) == {0, 1}


<details>
<summary>Follow-up answers</summary>

**What does one training example represent?**

One game card shown to one user in one request. The joined label records whether that user entered the shown experience within ten minutes.

**Are unshown items negative examples?**

No. Their outcomes are unobserved. Exposure and logging policy determine which labeled rows we see.

**What should you inspect before dropping missing rows?**

Check which columns are missing and what missingness means. Missing predictors do not automatically invalidate the label or the other fields.

</details>


## Problem 1 · Step 2 — Clean the data (4–9 min)

Create clean: remove exact duplicates and turn negative prior_genre_minutes into NaN. Join experiences into frame using a left many-to-one merge. Do not aggregate impressions away.

**Say before coding:** "A popular experience can appear many times, but its metadata must match only one row per impression."

**Possible interviewer questions**

1. Why not one_to_one?
2. Can we use minutes_after_impression?
3. Why postpone median imputation until after splitting?


In [4]:
clean = impressions.drop_duplicates().copy()
assert clean["impression_id"].is_unique
clean["prior_genre_minutes"] = clean["prior_genre_minutes"].mask(clean["prior_genre_minutes"].lt(0))
frame = clean.merge(experiences, on="experience_id", how="left", validate="many_to_one")


In [5]:
assert len(frame) == impressions["impression_id"].nunique()
assert frame["impression_id"].is_unique
assert frame["prior_genre_minutes"].dropna().ge(0).all()


<details>
<summary>Follow-up answers</summary>

**Why not one_to_one?**

Many impressions refer to the same experience; only the experience table key must be unique.

**Can we use minutes_after_impression?**

No. It is recorded after the prediction and directly exposes the outcome.

**Why postpone median imputation until after splitting?**

A median is learned from data. Fit it on training rows only; the held-out rows must not influence preprocessing.

</details>


## Problem 1 · Step 3 — Analyze and prepare features (9–14 min)

Create genre_summary with impression_count, joins and join_rate, grouped by genre. Also report join_rate by position. Explain why observed rates are not causal effects.

**Say before coding:** "The mean of a 0/1 outcome is its observed rate. Group counts tell us how much evidence each rate uses."

**Possible interviewer questions**

1. Why include counts next to rates?
2. Does higher join rate prove a genre is better?
3. How do you check that a merge did not multiply rows?


In [6]:
genre_summary = frame.groupby("genre", dropna=False).agg(
    impression_count=("impression_id", "size"),
    joins=("joined", "sum"),
    join_rate=("joined", "mean"),
)
display(genre_summary)
display(frame.groupby("position")["joined"].agg(["size", "mean"]))


,impression_count,joins,join_rate
genre,,,
adventure,605,54,0.089256
puzzle,947,84,0.088701
simulation,848,55,0.064858


,size,mean
position,,
1,600,0.115000
2,600,0.095000
3,600,0.058333
4,600,0.053333


In [7]:
assert genre_summary["impression_count"].sum() == len(frame)
np.testing.assert_allclose(genre_summary["join_rate"], genre_summary["joins"] / genre_summary["impression_count"])


<details>
<summary>Follow-up answers</summary>

**Why include counts next to rates?**

A rate from ten impressions is much less stable than a rate from ten thousand.

**Does higher join rate prove a genre is better?**

No. User mix, item quality, position and the existing recommendation policy can confound the comparison.

**How do you check that a merge did not multiply rows?**

Declare the expected key relationship with validate, then check row counts and the training-example identifier.

</details>


## Problem 2 · Step 4 — Select features and split (14–18 min)

Which columns would you use as features, which column is the target, and which fields would you exclude? We want to evaluate how well the model predicts joins for users it has never seen during training. What kind of train/test split would you choose, and why? Implement your choice and explain how you would verify that the split matches this evaluation goal.

**Say before coding:** "I will clarify the evaluation goal, choose a splitting strategy, and check that the resulting sets support that goal."

**Possible interviewer questions**

1. Would a random split of impression rows answer this evaluation question?
2. What would you check after splitting?
3. How do you check that X and y still correspond?


In [8]:
numeric_features = ["prior_genre_minutes", "friends_playing", "position", "experience_age_days"]
categorical_features = ["device", "genre"]
X = frame[numeric_features + categorical_features]
y = frame["joined"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=frame["user_id"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print("train/test positive fractions:", y_train.mean(), y_test.mean())


train/test positive fractions: 0.07708333333333334 0.09375


In [9]:
assert set(frame.iloc[train_idx]["user_id"]).isdisjoint(frame.iloc[test_idx]["user_id"])
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert y_train.nunique() == y_test.nunique() == 2
assert not {"user_id", "request_id", "experience_id", "impression_id", "joined", "minutes_after_impression"} & set(X.columns)


<details>
<summary>Follow-up answers</summary>

**Would a random split of impression rows answer this evaluation question?**

Not reliably: rows from one user can appear in both sets. For the stated unseen-user goal, keep all impressions from each user in only one set.

**What would you check after splitting?**

Check that user IDs do not overlap, X and y remain aligned, and each subset has enough positive and negative examples. A user-group split does not guarantee equal label proportions.

**How do you check that X and y still correspond?**

Select both from the same ordered table and apply the same split indices. Verify their indices match within each subset.

</details>


## Problem 2 · Step 5 — Preprocess and train (18–27 min)

Create numeric_pipeline with median imputation and StandardScaler, and categorical_pipeline with most-frequent imputation and OneHotEncoder(handle_unknown="ignore"). Combine them in preprocess using ColumnTransformer. Create model as a Pipeline with preprocess and LogisticRegression(max_iter=500), and baseline as DummyClassifier(strategy="prior"). Fit both using training data only.

**Say before coding:** "I will start with an interpretable pointwise probability model, not a complex recommender."

**Possible interviewer questions**

1. Should class_weight="balanced" always be on?
2. Why keep position for the classification exercise?
3. What happens when you call model.fit on the Pipeline?


In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
model = Pipeline([("preprocess", preprocess),
                  ("classifier", LogisticRegression(max_iter=500))])
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)


In [11]:
np.testing.assert_array_equal(model.classes_, [0, 1])
learned = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned, X_train[numeric_features].median().to_numpy())


<details>
<summary>Follow-up answers</summary>

**Should class_weight="balanced" always be on?**

No. It changes the effective training objective and probability interpretation. Compare business-relevant performance and calibration on validation data.

**Why keep position for the classification exercise?**

Position is known when evaluating response to an already displayed card. For ranking before placement, its dependence on the decision needs separate handling.

**What happens when you call model.fit on the Pipeline?**

The preprocessing steps fit and transform the training features. The final estimator then fits those transformed features against the training labels.

</details>


## Problem 2 · Step 6 — Evaluate (27–32 min)

Create prob=model.predict_proba(X_test)[:,1] and pred at threshold 0.5. Compare baseline and model on ROC-AUC, average precision, log loss, precision and recall at 0.5. Show the confusion matrix.

**Say before coding:** "A model can rank useful candidates well while producing few positive decisions at the default threshold."

**Possible interviewer questions**

1. Why might accuracy look good for a useless model?
2. Where should we choose a different threshold?
3. Can you keep changing the model after looking at test metrics?


In [12]:
prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)
def evaluate_binary(estimator):
    p = estimator.predict_proba(X_test)[:, 1]
    labels = (p >= 0.5).astype(int)
    return {"roc_auc": roc_auc_score(y_test, p),
            "average_precision": average_precision_score(y_test, p),
            "log_loss": log_loss(y_test, p, labels=[0, 1]),
            "precision@0.5": precision_score(y_test, labels, zero_division=0),
            "recall@0.5": recall_score(y_test, labels, zero_division=0)}
scores = pd.DataFrame({"prior": evaluate_binary(baseline),
                       "logistic": evaluate_binary(model)}).T
display(scores)
display(pd.DataFrame(confusion_matrix(y_test, pred, labels=[0, 1]),
    index=["actual: no join", "actual: join"],
    columns=["predicted: no join", "predicted: join"]))


,roc_auc,average_precision,log_loss,precision@0.5,recall@0.5
prior,0.500000,0.093750,0.312965,0.0,0.000000
logistic,0.688582,0.275121,0.288678,1.0,0.111111


,predicted: no join,predicted: join
actual: no join,435,0
actual: join,40,5


In [13]:
assert prob.shape == (len(y_test),)
assert ((prob >= 0) & (prob <= 1)).all()
np.testing.assert_allclose(scores.loc["prior", "average_precision"], y_test.mean())
np.testing.assert_allclose(scores.loc["prior", "roc_auc"], 0.5)


<details>
<summary>Follow-up answers</summary>

**Why might accuracy look good for a useless model?**

If joins are rare, predicting no join almost everywhere can achieve high accuracy.

**Where should we choose a different threshold?**

On a separate validation split, using a stated cost or precision/recall goal. Keep the final test set for one evaluation.

**Can you keep changing the model after looking at test metrics?**

Use a validation split or cross-validation for those choices. Repeatedly selecting against the test set makes its reported performance optimistic.

</details>


## Problem 2 · Step 7 — Predict and summarize (32–35 min)

Create new_cards with two rows containing all input features. Use the same assumed position=1 for both candidate cards and compute join_prob. Explain why score-based ranking still needs a separate evaluation.

**Say before coding:** "For comparing candidates I hold position constant, but these observational probabilities are not causal estimates."

**Possible interviewer questions**

1. Is global ROC-AUC the same as ranking quality per request?
2. Can this model score every unseen experience?
3. What must training and inference share?


In [14]:
new_cards = pd.DataFrame([
    {"prior_genre_minutes": 80, "friends_playing": 2, "position": 1,
     "experience_age_days": 200, "device": "mobile", "genre": "adventure"},
    {"prior_genre_minutes": 10, "friends_playing": 0, "position": 1,
     "experience_age_days": 20, "device": "mobile", "genre": "puzzle"},
])
join_prob = model.predict_proba(new_cards)[:, 1]
display(new_cards.assign(join_probability=join_prob))


,prior_genre_minutes,friends_playing,position,experience_age_days,device,genre,join_probability
0,80,2,1,200,mobile,adventure,0.394419
1,10,0,1,20,mobile,puzzle,0.044811


In [15]:
assert join_prob.shape == (2,)
assert np.isfinite(join_prob).all()


<details>
<summary>Follow-up answers</summary>

**Is global ROC-AUC the same as ranking quality per request?**

No. It compares positive and negative rows globally. Ranking evaluation compares candidates within a request using metrics such as Recall@K or NDCG@K.

**Can this model score every unseen experience?**

Only if the required content and historical features are available. A full system also needs candidate retrieval, safety filtering and online evaluation.

**What must training and inference share?**

The feature definitions, column meanings and fitted preprocessing. Reuse the saved pipeline rather than relearning imputation, scaling or categories.

</details>


## If the interviewer extends the problem

**Optional ranking follow-up (not in the 35-minute core):** score held-out requests with position held at 1, sort each request by score, and compute Recall@2 over requests with at least one observed join. State that these are outcomes under the old display policy, not unbiased counterfactual labels for a new ordering. Report how many requests were excluded because their denominator is zero. This diagnostic is not proof that re-ranking will improve joins.


In [16]:
ranking_X = X_test.copy()
ranking_X["position"] = 1
ranked = frame.iloc[test_idx][["request_id", "experience_id", "joined"]].copy()
ranked["score"] = model.predict_proba(ranking_X)[:, 1]
ranked = ranked.sort_values(["request_id", "score", "experience_id"], ascending=[True, False, True])
top_two = ranked.groupby("request_id", sort=False).head(2)
positives = ranked.groupby("request_id")["joined"].sum()
hits = top_two.groupby("request_id")["joined"].sum().reindex(positives.index, fill_value=0)
eligible = positives.gt(0)
recall_at_2 = (hits.loc[eligible] / positives.loc[eligible]).mean()
print("Observed-label Recall@2:", recall_at_2)
print("Eligible requests:", eligible.sum(), "excluded no-positive requests:", (~eligible).sum())
assert 0 <= recall_at_2 <= 1


Observed-label Recall@2: 0.6923076923076923
Eligible requests: 39 excluded no-positive requests: 81


## Scope and sources

These are **original synthetic exercises**, not reported Roblox questions and not official CodeSignal assessments. The user-supplied **Data Analysis & Modeling Coding Interview** guide defines the format: 5 minutes intro, 35 minutes for two problems (analysis and modeling), 5 minutes Q&A; pandas, NumPy, sklearn and SciPy are available. The per-step times here are suggested practice checkpoints, not official grading criteria.

The guide does not identify a team or prescribe regression/softmax topics. We chose those to practice task-appropriate modeling and metrics. Business themes are informed by the [official PhD Early Career MLE posting](https://careers.roblox.com/jobs/8242623), which mentions engagement, safety and recommendation systems. Source checked 2026-10-22; exact interview questions are unknown.

Implementation references: [sklearn 1.6 LogisticRegression](https://scikit-learn.org/1.6/modules/generated/sklearn.linear_model.LogisticRegression.html), [model evaluation](https://scikit-learn.org/1.6/modules/model_evaluation.html), [data leakage](https://scikit-learn.org/1.6/common_pitfalls.html#data-leakage).
